# part A

In [1]:

import numpy as np
import pandas as pd


col_names = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight", "Viscera_weight",
    "Shell_weight", "Rings"
]

df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data",
    header=None,
    names=col_names
)

print("Number of rows:", len(df))
print("Column names:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())


Number of rows: 4177
Column names: ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings']

First 5 rows:
  Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  


*  input:  the physical measurements of the abalone (length, diameter, height, weights)
*  output: age of the abalone, estimated as Rings + 1.5
* output is numeric because age is a continuous quantity (years), not a category, so regression makes sense


 # A3
 * selected 3 features are :Shell_weight, Diameter, Height
  *  Shell_weight: shell grows as the animal ages, so it's a strong
   indicator of age. heavier shell = older abalone generally.
* Diameter: overall body size scales with age. diameter is a clean
             single measurement and avoids redundancy with length.
 * Height: captures the 3D growth of the shell. an abalone that has
             grown taller has likely been alive longer.

In [2]:

df["Age"] = df["Rings"] + 1.5
selected_features = ["Shell_weight", "Diameter", "Height"]

X = df[selected_features].values
y = df["Age"].values.reshape(-1, 1)


In [3]:
# A4. train-test split (80/20, no sklearn)

np.random.seed(42)
total_samples = len(X)
indices = np.arange(total_samples)
np.random.shuffle(indices)

split_point = int(0.8 * total_samples)
train_idx = indices[:split_point]
test_idx  = indices[split_point:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("\nShapes after train-test split:")
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)



Shapes after train-test split:
X_train: (3341, 3)
X_test:  (836, 3)
y_train: (3341, 1)
y_test:  (836, 1)


# A5. Normalize Inputs

In [4]:

train_mean = X_train.mean(axis=0)
train_std  = X_train.std(axis=0)

X_train_norm = (X_train - train_mean) / train_std
X_test_norm  = (X_test  - train_mean) / train_std


*  why normalization is needed for learning:
Normalization is important because if features have very different scales, gradient descent updates become uneven and training becomes unstable.
By scaling features to mean 0 and standard deviation 1, learning becomes faster and more stable.

# part b

In [5]:

def forward(X, w, b):
    """
    Computes y_hat = X @ w + b
    X shape: (N, d)
    w shape: (d, 1)
    b shape: scalar
    returns y_hat shape: (N, 1)
    """
    y_hat = np.dot(X, w) + b
    return y_hat

d = X_train_norm.shape[1]
w_init = np.zeros((d, 1))          # (3, 1)
b_init = 0.0

y_hat_sample = forward(X_train_norm, w_init, b_init)

print("\n-- Shape Check (Part B) --")
print("X shape:     ", X_train_norm.shape)
print("w shape:     ", w_init.shape)
print("b:           ", b_init, " (scalar)")
print("y_hat shape: ", y_hat_sample.shape)

# Checkpoint:
# parameters are: w ,the weight vector and b ,the bias
# no. of parameters: d + 1 = 3 + 1 = 4



-- Shape Check (Part B) --
X shape:      (3341, 3)
w shape:      (3, 1)
b:            0.0  (scalar)
y_hat shape:  (3341, 1)


# part c

In [6]:
def mse(y, y_hat):
    """
    Mean Squared Error:
    L = (1/N) * sum( (y_i - y_hat_i)^2 )
    """
    N = len(y)
    loss = (1 / N) * np.sum((y - y_hat) ** 2)
    return loss

# checkpoint
* why square: makes all errors positive and punishes big errors more
*  what mistakes are expensive: large errors, e.g. being off by 6 costs 36, off by 2 costs only 4

# part d

In [7]:


# Checkpoint:
# what gradient means in words: it's the slope of the loss
# tells us which direction makes loss go up
# why subtracting gradient reduces loss:
# moving opposite to the slope always goes downhill


# implementing gradients ----

def grad_w(X, y, y_hat):
    """
    Gradient of MSE loss with respect to w.

    Derivation:
    L = (1/N) * sum( (y_hat - y)^2 )
    dL/dw = (2/N) * X^T @ (y_hat - y)

    The 2 is often absorbed into the learning rate, but I'll keep it
    here for correctness. shape returned: (d, 1) to match w.
    """
    N = len(y)
    error = y_hat - y           # shape: (N, 1)
    dW = (2 / N) * np.dot(X.T, error)  # shape: (d, 1)
    return dW

def grad_b(y, y_hat):
    """
    Gradient of MSE loss with respect to b.

    dL/db = (2/N) * sum(y_hat - y)

    Returns a scalar to match b.
    """
    N = len(y)
    error = y_hat - y           # shape: (N, 1)
    db = (2 / N) * np.sum(error)
    return db

w_check = np.zeros((d, 1))
b_check = 0.0
yhat_check = forward(X_train_norm, w_check, b_check)
dW_check = grad_w(X_train_norm, y_train, yhat_check)
db_check = grad_b(y_train, yhat_check)
print("\n--- Gradient Shape Check ---")
print("dW shape:", dW_check.shape, " should match w shape:", w_check.shape)
print("db type:", type(db_check), " (scalar, matches b)")

# Checkpoint:
# meaning of large gradient:
# the loss is very sensitive to that weight, we're far from optimal
# effect of too-large learning rate:
# we overshoot the minimum and loss can explode instead of decreasing




--- Gradient Shape Check ---
dW shape: (3, 1)  should match w shape: (3, 1)
db type: <class 'numpy.float64'>  (scalar, matches b)


#  part E

In [8]:

np.random.seed(0)
w = np.random.randn(d, 1) * 0.01
b = 0.0                              # bias starts at zero

learning_rate = 0.01
epochs = 1000
print_every = 100


print("\n-- Training Loop --")
loss_history = []

for epoch in range(epochs):
    # 1) forward pass
    y_hat = forward(X_train_norm, w, b)

    #  compute loss
    loss = mse(y_train, y_hat)
    loss_history.append(loss)

    #  compute gradients
    dW = grad_w(X_train_norm, y_train, y_hat)
    db = grad_b(y_train, y_hat)

    #  update w and b (gradient descent step)
    w = w - learning_rate * dW
    b = b - learning_rate * db

    if epoch % print_every == 0:
        print(f"Epoch {epoch:4d}  |  Train MSE: {loss:.4f}")






-- Training Loop --
Epoch    0  |  Train MSE: 141.4100
Epoch  100  |  Train MSE: 8.8355
Epoch  200  |  Train MSE: 6.5289
Epoch  300  |  Train MSE: 6.4596
Epoch  400  |  Train MSE: 6.4401
Epoch  500  |  Train MSE: 6.4279
Epoch  600  |  Train MSE: 6.4199
Epoch  700  |  Train MSE: 6.4145
Epoch  800  |  Train MSE: 6.4109
Epoch  900  |  Train MSE: 6.4084


# part F

In [9]:

# final pred on test set
y_hat_test = forward(X_test_norm, w, b)

# MSE
test_mse = mse(y_test, y_hat_test)
N_test = len(y_test)
test_mae = (1 / N_test) * np.sum(np.abs(y_test - y_hat_test))

print("\n--- Test Evaluation ---")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test MAE: {test_mae:.4f}")

print("\n 5 sample pred --")
print(f"{'True Age':>10}  {'Predicted Age':>14}  {'Absolute Error':>15}")
for i in range(5):
    true_val  = float(y_test[i])
    pred_val  = float(y_hat_test[i])
    abs_error = abs(true_val - pred_val)
    print(f"{true_val:>10.2f}  {pred_val:>14.2f}  {abs_error:>15.2f}")

# Checkpoint:
# systematic errors:
# model struggles with very young and very old abalones
# observed bias:
# predictions cluster near the average age, extreme values get pulled toward the middle

print("\n--- Final Learned Parameters ---")
print("weights (w):")
for i, feat in enumerate(selected_features):
    print(f"  w_{i+1} ({feat}): {w[i][0]:.4f}")
print(f"  b (bias): {b:.4f}")


--- Test Evaluation ---
Test MSE: 5.5886
Test MAE: 1.7685

 5 sample pred --
  True Age   Predicted Age   Absolute Error
     10.50            9.59             0.91
     11.50           12.91             1.41
     10.50           11.73             1.23
     11.50            9.33             2.17
      7.50            8.60             1.10

--- Final Learned Parameters ---
weights (w):
  w_1 (Shell_weight): 1.7073
  w_2 (Diameter): 0.0570
  w_3 (Height): 0.4065
  b (bias): 11.4315


/tmp/ipykernel_391/3091572489.py:16: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  true_val  = float(y_test[i])
/tmp/ipykernel_391/3091572489.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pred_val  = float(y_hat_test[i])
